# HW3 - PyTorch Tutorial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

torch.manual_seed(0)
print("torch version:", torch.__version__)
print("accelerator available:", torch.accelerator.is_available())

## Part 1: Tensors

A **tensor** is PyTorch's basic data structure: a multi-dimensional array that holds a model's inputs, outputs and parameters. This part is a written summary of what I learned. It compares tensors with MATLAB matrices and NumPy arrays and explains why tensors are useful for neural networks.

### Tensors are like MATLAB matrices

After reading, I feel that tensor operations look a lot like MATLAB matrix operations. Both distinguish an **element-wise** product from a true **matrix** product, and both have slicing, concatenation, transpose, and functions that create arrays of zeros, ones or random numbers.

After searching information, I found PyTorch copied its tensor API from NumPy, and NumPy was itself designed to feel like MATLAB. Therefore, it wasn't an coincidence.

In [ ]:
# element-wise product vs. matrix product
tensor = torch.ones(4, 4)
tensor[:, 3] = 0

print(tensor * tensor)      # element-wise
print(tensor @ tensor.T)    # matrix product

### Tensor vs. NumPy array

(Table made by AI)
| | NumPy array | PyTorch tensor |
|---|---|---|
| Where it runs | CPU only | CPU or a GPU |
| Gradients | not supported | can track operations and compute gradients automatically |
| Default float type | `float64` | `float32` |
| Designed for | general numerical computing | training neural networks |

### Why tensors are useful

1. **Automatic differentiation.**
2. **GPU acceleration.**
3. **Built for neural networks.**
4. **Interoperability.** 

## Part 2: A Gentle Introduction to `torch.autograd`

`torch.autograd` is PyTorch's automatic differentiation engine. Training a network has two steps: 
**forward propagation** make a prediction
**backward propagation** compute the gradient of the error with respect to every parameter, then update the parameters. This part is about how we can use this tool to calculate and update the parameters.

### Computational graph and exclusion from it

Autograd records every operation in a directed acyclic graph (DAG) during the forward pass and walks it backwards with the chain rule. A tensor with `requires_grad=False` is left out of the graph. An output requires gradients if any of its inputs does.

### Gradient calculation and parameter update

**Gradient.** The gradient of the loss $L$ with respect to a parameter $w$ is $\partial L/\partial w$. It is the loss changes when $w$ changes a little bit. We can use chain rule to get the gradient. For example, if $L$ depends on $\hat{y}$ and $\hat{y}$ depends on $w$, then

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial w}$$

**Parameter update.** After the gradients are calculated, the optimizer updates every parameter. The simplest rule is stochastic gradient descent (SGD):

$$w_{\text{new}} = w_{\text{old}} - \eta \cdot \frac{\partial L}{\partial w}$$

Here $\eta$ is the learning rate.


In [ ]:
# gradients with autograd
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

Q = 3 * a**3 - b**2
Q.backward(gradient=torch.tensor([1., 1.]))

print(a.grad)    # 9 * a**2
print(b.grad)    # -2 * b

## Part 3: Neural Networks

Networks are built with `torch.nn`. To define an `nn.Module`, we need to define its layers and a forward method. A layer is an object that contains learnable parameters. The forward method specifies the order in which data passes from one layer to the next, and the shapes between layers must match.

Different types of data call for different neural network architectures, such as MLP, CNN, RNN, and Transformer. The example uses a simple CNN for digit recognition.

### Loss function

A loss function compares the output with the target. Here: `nn.MSELoss`, the mean squared error.

### Update the weights

This part is introduced above. The simplest update rule is stochastic gradient descent: `weight = weight - learning_rate * gradient`. `torch.optim` implements this and other rules. 

## Part 4: Training a Classifier

In this part, the tutorial train an image classifier on CIFAR-10 (10 classes of 3x32x32 color images). Steps:

1. Load and normalize the CIFAR-10 training and test datasets with `torchvision`
2. Define a convolutional neural network
3. Define a loss function and optimizer
4. Train the network on the training data
5. Test the network on the test data

### 2. Define a convolutional neural network

This is the same network as in Part 3, a simple CNN. The only difference is the input. In Part 3 the images were grayscale, so they had 1 channel. The CIFAR-10 images are color (RGB) images, so they have 3 channels. Because of this, the first convolution layer takes 3 input channels instead of 1. Everything else is the same.

For one 3x32x32 image, the network does the following:

1. `conv1`: convolution with 6 filters of size 5x5, then ReLU, then 2x2 max pooling. The size changes 3x32x32 -> 6x28x28 -> 6x14x14.
2. `conv2`: convolution with 16 filters of size 5x5, then ReLU, then 2x2 max pooling. The size changes 6x14x14 -> 16x10x10 -> 16x5x5.
3. Flatten: the 16x5x5 output becomes one vector of 400 numbers.
4. `fc1`: fully connected layer, 400 -> 120, then ReLU.
5. `fc2`: fully connected layer, 120 -> 84, then ReLU.
6. `fc3`: fully connected layer, 84 -> 10. The 10 outputs are the scores of the 10 classes, and the class with the highest score is the prediction.


### Training on the GPU

A network is moved to the GPU like a tensor: `net.to(device)`, and the inputs and labels must be moved at every step too: `inputs, labels = data[0].to(device), data[1].to(device)`. On my Mac the device is Apple's `mps`.

[1] Language was polished using Claude Sonnet 5.5 (via Claude Code), October 5-6, 2026, Anthropic, https://claude.com/claude-code. The same tool was used to answer my questions about the tutorial (for example how tensors compare with MATLAB matrices, and how forward and backward propagation work). The table in Part 1 was made by AI. 